# 05 — SFT Training (Distilled Dataset)

Fine-tunes Cosmos-Reason2-2B on the 8B-distilled underwater VQA dataset using QLoRA.

**Run order:** Cell 1 (install) → Cell 2 (mount + login + download videos) → Cell 3 (dataset class + processor) → Cell 4 (smoke test) → Cell 5 (full training) → Cell 6 (save + upload) → Cell 7 (MCQ eval)

## Cell 1 — Install dependencies

In [ ]:
!pip install -Uq "trl[peft]==0.26.1" "bitsandbytes==0.49.0" "tensorboard==2.20.0" decord fiftyone

## Cell 2 — Mount Drive, Login & Download Videos

Mounts Drive, logs into HuggingFace, and downloads the WebUOT-238-Test videos via fiftyone.
Video path is discovered dynamically so it works regardless of fiftyone version.

In [ ]:
from google.colab import drive, userdata
from huggingface_hub import login

drive.mount('/content/drive', force_remount=True)
login(token=userdata.get("HF_TOKEN"))

import os, json, torch, numpy as np
from PIL import Image
import decord
import fiftyone as fo
from fiftyone.utils.huggingface import load_from_hub

# ── Paths ─────────────────────────────────────────────────────────────────────
TRAIN_PATH      = "/content/drive/MyDrive/cosmos-cookoff/data/underwater_vqa_train_distilled.json"
VAL_PATH        = "/content/drive/MyDrive/cosmos-cookoff/data/underwater_vqa_val_distilled.json"
OUTPUT_DIR      = "/content/drive/MyDrive/cosmos-cookoff/outputs/sft_checkpoint_distilled"
FINAL_MODEL_DIR = "/content/drive/MyDrive/cosmos-cookoff/outputs/final_model_distilled"

for path in [TRAIN_PATH, VAL_PATH]:
    print(f"{'✓' if os.path.exists(path) else '✗'} {path}")

with open(TRAIN_PATH) as f:
    train_data = json.load(f)
with open(VAL_PATH) as f:
    val_data = json.load(f)

print(f"Train: {len(train_data)} pairs")
print(f"Val:   {len(val_data)} pairs")

# ── Download videos & discover real path ─────────────────────────────────────
print("\nLoading WebUOT-238-Test via fiftyone...")
fo_dataset = load_from_hub("Voxel51/WebUOT-238-Test", overwrite=False)

# Discover the actual video directory from fiftyone's stored filepath
VIDEO_DIR = os.path.dirname(fo_dataset.first().filepath)
print(f"Video directory: {VIDEO_DIR}")
print(f"Files present:   {len(os.listdir(VIDEO_DIR))}")

def fix_path(p):
    """Remap any stored path to the local fiftyone video copy."""
    return os.path.join(VIDEO_DIR, os.path.basename(p))

# Patch JSON files to use correct local paths
for json_path in [TRAIN_PATH, VAL_PATH]:
    with open(json_path) as f:
        data = json.load(f)
    for item in data:
        item["video"] = fix_path(item["video"])
    with open(json_path, "w") as f:
        json.dump(data, f, indent=2)
    print(f"Patched {len(data)} entries in {os.path.basename(json_path)}")

# Reload after patching
with open(TRAIN_PATH) as f:
    train_data = json.load(f)
with open(VAL_PATH) as f:
    val_data = json.load(f)

sample_video = train_data[0]["video"]
print(f"\nSample path:   {sample_video}")
print(f"Sample exists: {os.path.exists(sample_video)}")

## Cell 3 — Dataset class + processor

In [ ]:
from torch.utils.data import Dataset as TorchDataset
from transformers import AutoProcessor

MODEL_NAME    = "nvidia/Cosmos-Reason2-2B"
N_FRAMES      = 8
SYSTEM_PROMPT = (
    "You are a helpful assistant specializing in underwater scene understanding "
    "for autonomous underwater vehicle (AUV) navigation."
)

print("Loading processor...")
processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
processor.image_processor.min_pixels = 256 * 28 * 28
processor.image_processor.max_pixels = 512 * 28 * 28
print("Processor loaded.")


def extract_frames(video_path, n_frames=N_FRAMES):
    vr      = decord.VideoReader(video_path, ctx=decord.cpu(0))
    indices = np.linspace(0, len(vr) - 1, n_frames, dtype=int)
    frames  = vr.get_batch(indices).asnumpy()
    return [Image.fromarray(frames[i]) for i in range(n_frames)]


class UnderwaterVQADataset(TorchDataset):
    def __init__(self, json_path, processor, max_samples=None, n_frames=N_FRAMES):
        with open(json_path) as f:
            self.data = json.load(f)
        if max_samples:
            self.data = self.data[:max_samples]
        self.processor = processor
        self.n_frames  = n_frames
        print(f"Dataset: {len(self.data)} samples | {n_frames} frames/video")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item     = self.data[idx]
        question = item["conversations"][0]["value"].replace("<video>" + chr(10), "").strip()
        answer   = item["conversations"][1]["value"]

        try:
            frames = extract_frames(item["video"], self.n_frames)
        except Exception:
            frames = [Image.new("RGB", (224, 224), (0, 0, 0))] * self.n_frames

        messages_full = [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": [
                *[{"type": "image", "image": f} for f in frames],
                {"type": "text",   "text": question},
            ]},
            {"role": "assistant", "content": answer},
        ]
        messages_prefix = messages_full[:-1]

        full_text = self.processor.apply_chat_template(
            messages_full, tokenize=False, add_generation_prompt=False
        )
        inputs = self.processor(
            text=[full_text], images=frames, return_tensors="pt", padding=False
        )

        prefix_text = self.processor.apply_chat_template(
            messages_prefix, tokenize=False, add_generation_prompt=True
        )
        prefix_inputs = self.processor(
            text=[prefix_text], images=frames, return_tensors="pt", padding=False
        )

        input_ids      = inputs["input_ids"][0]
        attention_mask = inputs["attention_mask"][0]
        prefix_len     = prefix_inputs["input_ids"].shape[1]

        labels = input_ids.clone()
        labels[:prefix_len] = -100

        result = {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}
        for k, v in inputs.items():
            if k not in result:
                result[k] = v[0]
        return result


print("Dataset class defined.")

## Cell 4 — Dataset smoke test (2 samples)

Verify videos load correctly before committing to full training.

In [ ]:
test_ds = UnderwaterVQADataset(TRAIN_PATH, processor, max_samples=2)
sample  = test_ds[0]

print("Keys:", list(sample.keys()))
print("input_ids shape:", sample["input_ids"].shape)
print("Supervised tokens:", (sample["labels"] != -100).sum().item())

supervised_ids = sample["input_ids"][sample["labels"] != -100]
decoded = processor.decode(supervised_ids[:50], skip_special_tokens=True)
print(f"First 50 supervised tokens: {repr(decoded)}")

# Sanity check — supervised tokens should be > 1 (not just punctuation)
assert (sample["labels"] != -100).sum().item() > 1, "ERROR: Almost no supervised tokens — check video loading"
print("Smoke test passed ✓")

## Cell 5 — Full training

**Distilled dataset** — open-ended answers generated by Cosmos-Reason2-8B, MCQ labels from dataset annotations.

Expected: ~40–50 min on H100. Checkpoints saved every 20 steps to Drive.

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from transformers import BitsAndBytesConfig, Qwen3VLForConditionalGeneration
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

os.makedirs(OUTPUT_DIR,      exist_ok=True)
os.makedirs(FINAL_MODEL_DIR, exist_ok=True)

model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    dtype="auto",
    device_map="auto",
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    ),
)

peft_config = LoraConfig(
    r=32,
    lora_alpha=32,
    target_modules=["down_proj", "o_proj", "k_proj", "q_proj",
                    "gate_proj", "up_proj", "v_proj"],
)

full_train_ds = UnderwaterVQADataset(TRAIN_PATH, processor, max_samples=None)

training_args = SFTConfig(
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_steps=50,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    optim="adamw_8bit",
    max_length=None,
    output_dir=OUTPUT_DIR,
    logging_steps=10,
    save_steps=20,
    save_total_limit=3,
    report_to="tensorboard",
    remove_unused_columns=False,
    dataset_kwargs={"skip_prepare_dataset": True},
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_num_workers=2,
    dataloader_prefetch_factor=2,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=full_train_ds,
    peft_config=peft_config,
)

print(f"Training on {len(full_train_ds)} samples x 3 epochs")
print(f"GPU:  {torch.cuda.get_device_properties(0).name}")
print(f"VRAM: {torch.cuda.memory_reserved()/1024**3:.1f} GB")
print("Starting...")

stats = trainer.train()

peak_vram  = round(torch.cuda.max_memory_reserved()/1024**3, 2)
runtime_hr = round(stats.metrics["train_runtime"]/3600, 2)

print(f"TRAINING COMPLETE")
print(f"Time:       {runtime_hr} hours")
print(f"Peak VRAM:  {peak_vram} GB")
print(f"Final loss: {stats.metrics.get('train_loss')}")

## Cell 6 — Save model to Drive & upload to HuggingFace Hub

In [ ]:
from huggingface_hub import HfApi

trainer.save_model(FINAL_MODEL_DIR)
processor.save_pretrained(FINAL_MODEL_DIR)

metadata = {
    "base_model":     MODEL_NAME,
    "training_pairs": len(full_train_ds),
    "epochs":         3,
    "learning_rate":  2e-4,
    "n_frames":       N_FRAMES,
    "lora_r":         32,
    "lora_alpha":     32,
    "peak_vram_gb":   peak_vram,
    "train_hours":    runtime_hr,
    "final_loss":     stats.metrics.get("train_loss"),
    "dataset":        "WebUOT-238-Test distilled via Cosmos-Reason2-8B",
    "distillation":   "8B teacher generated open-ended answers for scene/hazard/obj/spatial types",
    "task":           "Underwater VQA for AUV Navigation",
}

import json as _json
with open(f"{FINAL_MODEL_DIR}/training_metadata.json", "w") as f:
    _json.dump(metadata, f, indent=2)

print(f"Model saved to: {FINAL_MODEL_DIR}")
for k, v in metadata.items():
    print(f"  {k}: {v}")

# ── Upload to HuggingFace Hub ─────────────────────────────────────────────────
HF_REPO_ID = "15juneee/cosmos-reason2-underwater-auv"

api = HfApi()
api.create_repo(repo_id=HF_REPO_ID, repo_type="model", exist_ok=True)
api.upload_folder(
    folder_path=FINAL_MODEL_DIR,
    repo_id=HF_REPO_ID,
    repo_type="model",
)
print(f"\nUploaded to: https://huggingface.co/{HF_REPO_ID}")

## Cell 7 — Quick 30-sample MCQ evaluation

Spot-check fine-tuned model accuracy on held-out MCQ samples.

In [ ]:
import random
random.seed(42)

# Reload val data (paths already patched in Cell 2)
with open(VAL_PATH) as f:
    val_data = json.load(f)

mcq_samples = [x for x in val_data if "_mcq_" in x["id"] or "_cat_mcq_" in x["id"]]
mcq_subset  = random.sample(mcq_samples, 30)

eval_model = trainer.model
eval_model.eval()

correct = 0
skipped = 0

for i, sample in enumerate(mcq_subset):
    question = sample["conversations"][0]["value"].replace("<video>" + chr(10), "").strip()
    gt       = sample["conversations"][1]["value"].strip().upper()

    try:
        frames = extract_frames(sample["video"])
    except Exception as e:
        skipped += 1
        print(f"  SKIPPED [{i+1}] {sample['id']}: {e}")
        continue

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": [
            *[{"type": "image", "image": f} for f in frames],
            {"type": "text",   "text": question},
        ]},
    ]
    text   = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=frames, return_tensors="pt", padding=True)
    inputs = {k: v.to(eval_model.device) for k, v in inputs.items()}

    with torch.no_grad():
        out_ids = eval_model.generate(**inputs, max_new_tokens=16, do_sample=False)

    generated = out_ids[0][inputs["input_ids"].shape[1]:]
    pred      = processor.decode(generated, skip_special_tokens=True).strip().upper()

    # Extract just the letter (A/B/C/D) from both GT and prediction
    gt_letter   = gt.split(")")[0].strip()   if ")" in gt   else gt[0]
    pred_letter = pred.split(")")[0].strip() if ")" in pred else pred[0]
    is_correct  = gt_letter == pred_letter

    if is_correct:
        correct += 1
    print(f"{i+1}/30 | GT: {gt[:20]} | PRED: {pred[:20]} | {chr(10003) if is_correct else chr(10007)}")

evaluated = 30 - skipped
print(f"\nFine-tuned MCQ:    {correct}/{evaluated} = {correct/evaluated*100:.1f}%")
print(f"Zero-shot baseline: 2/10 = 20.0%")
if skipped:
    print(f"Skipped (video error): {skipped}")